# ICM-UADE — Mapa de precios por sucursal
**INECO · Universidad Argentina de la Empresa**

Genera el mapa interactivo, rankings de cadenas, provincias y barrios CABA.

▶️ **Ejecutar todo** (`Runtime → Run all`) y esperar ~10 minutos.
La única celda modificable es la primera — para elegir el mes.

In [ ]:
# ══════════════════════════════════════════════════════════════════
# CONFIGURACIÓN — única celda que podés modificar
# ══════════════════════════════════════════════════════════════════

# Mes a analizar (formato 'YYYY-MM').
# Dejalo en None para usar el último mes disponible automáticamente.
MES = None   # ← cambiá a '2026-04' si querés un mes específico

# ── No modificar nada de acá para abajo ───────────────────────────
REPO_URL     = 'https://github.com/santiagoriverti/analisis_precios_minoristas_UADE.git'
MANIFEST_URL = 'https://raw.githubusercontent.com/santiagoriverti/analisis_precios_minoristas_UADE/main/data/drive_manifest.json'
DIR_DATOS    = '/content/datos_sepa'
DIR_REPO     = '/content/repo'

# IDs de las carpetas de Drive de INECO (no modificar)
FOLDER_HISTORICO = '13GONeBs5lQCSUdBioHYk-8GhfDtIyliD'  # 2018-2023
FOLDER_RECIENTE  = '1GNs9SrZ4BIoBsviBVWYYqRcsj4dwPF-I'  # 2024-presente

NOMBRE_MAESTRO_PROD = 'Maestro de Productos Interno.xlsx'
NOMBRE_MAESTRO_SUC  = 'maestro_sucursales_completo.xlsx'

In [ ]:
# ══════════════════════════════════════════════════════════════════
# PASO 1 — Instalación y clonado
# ══════════════════════════════════════════════════════════════════
import os, sys

if not os.path.exists(DIR_REPO):
    os.system(f'git clone -q {REPO_URL} {DIR_REPO}')
    print('✓ Repositorio clonado')
else:
    os.system(f'git -C {DIR_REPO} pull -q')
    print('✓ Repositorio actualizado')

os.system('pip install -q gdown folium branca openpyxl pyarrow')
sys.path.insert(0, f'{DIR_REPO}/src')
os.makedirs(DIR_DATOS, exist_ok=True)
print('✓ Listo')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# PASO 2 — Detectar método de descarga
#
# Intenta usar el manifest (sin auth).
# Si el manifest no está poblado, usa la API de Drive (pide auth una vez).
# ══════════════════════════════════════════════════════════════════
import requests, json, re

manifest = requests.get(MANIFEST_URL).json()

# Meses con IDs reales en el manifest
meses_manifest = {
    m: v for m, v in manifest.get('meses', {}).items()
    if isinstance(v, dict)
    and v.get('parte1_id', 'COMPLETAR') not in ('COMPLETAR', '', None)
    and not m.startswith('_')
}

USAR_MANIFEST = len(meses_manifest) > 0

if USAR_MANIFEST:
    print(f'✓ Manifest cargado — {len(meses_manifest)} meses disponibles')
    MES_ANALIZAR = MES if (MES and MES in meses_manifest) else max(meses_manifest.keys())
    if MES and MES not in meses_manifest:
        print(f'  ⚠ {MES} no está en el manifest — usando el último: {MES_ANALIZAR}')
    print(f'  Mes seleccionado: {MES_ANALIZAR}')
else:
    print('ℹ El manifest aún no tiene IDs cargados.')
    print('  Se usará la API de Google Drive → vas a ver un popup de autorización.')
    MES_ANALIZAR = MES  # puede ser None, se detecta en el siguiente paso

In [ ]:
# ══════════════════════════════════════════════════════════════════
# PASO 3 — Descargar archivos
# ══════════════════════════════════════════════════════════════════
from pathlib import Path

archivos_locales = {}  # se llena en 3a o 3b

# ── 3a: manifest disponible → gdown, sin autenticación ───────────
if USAR_MANIFEST:
    import gdown

    datos_mes = meses_manifest[MES_ANALIZAR]
    datos_maestros = manifest.get('maestros', {})

    mm   = MES_ANALIZAR[5:]   # '04'
    aaaa = MES_ANALIZAR[:4]   # '2026'
    MES_MMAAAA = f'{mm}{aaaa}'  # '042026'

    descargas = [
        (datos_mes.get('parte1_id'), f'{DIR_DATOS}/{MES_MMAAAA}_pais_parte1COMPLETO.csv.gz', 'SEPA parte 1'),
        (datos_mes.get('parte2_id'), f'{DIR_DATOS}/{MES_MMAAAA}_pais_parte2COMPLETO.csv.gz', 'SEPA parte 2'),
        (datos_maestros.get('productos_id'),  f'{DIR_DATOS}/{NOMBRE_MAESTRO_PROD}', 'Maestro productos'),
        (datos_maestros.get('sucursales_id'), f'{DIR_DATOS}/{NOMBRE_MAESTRO_SUC}',  'Maestro sucursales'),
    ]

    for file_id, output_path, label in descargas:
        p = Path(output_path)
        if p.exists() and p.stat().st_size > 100_000:
            print(f'  ↩ {label} ya descargado ({p.stat().st_size/1e6:.0f} MB)')
        else:
            print(f'  ↓ {label}...')
            gdown.download(id=file_id, output=output_path, quiet=False, fuzzy=True)
        archivos_locales[label] = output_path

# ── 3b: manifest vacío → Drive API con autenticación ─────────────
else:
    from google.colab import auth
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaIoBaseDownload
    import io

    print('Autorizando acceso a Google Drive...')
    auth.authenticate_user()
    drive = build('drive', 'v3', cache_discovery=False)
    print('✓ Autorizado')

    # Listar archivos recursivamente (incluye subcarpetas)
    def listar_carpeta(folder_id, _nivel=0):
        archivos, subcarpetas, page_token = [], [], None
        while True:
            r = drive.files().list(
                q=f"'{folder_id}' in parents and trashed=false",
                fields='nextPageToken,files(id,name,size,mimeType)',
                pageSize=1000, pageToken=page_token
            ).execute()
            for item in r.get('files', []):
                if item.get('mimeType') == 'application/vnd.google-apps.folder':
                    subcarpetas.append(item)
                else:
                    archivos.append(item)
            page_token = r.get('nextPageToken')
            if not page_token:
                break
        indent = '  ' * (_nivel + 1)
        for sub in subcarpetas:
            print(f'{indent}↳ subcarpeta: {sub["name"]}')
            archivos.extend(listar_carpeta(sub['id'], _nivel + 1))
        return archivos

    print('Listando archivos en Drive...')
    todos = {}
    meses_drive = {}
    for fid in (FOLDER_HISTORICO, FOLDER_RECIENTE):
        for a in listar_carpeta(fid):
            todos[a['name']] = a
            m = re.match(r'(\d{2})(\d{4})_pais_parte(\d)COMPLETO\.csv\.gz', a['name'])
            if m:
                mes_fmt = f'{m.group(2)}-{m.group(1)}'
                meses_drive.setdefault(mes_fmt, {})[f'parte{m.group(3)}'] = a

    # Diagnóstico de qué se encontró
    csvgz = sorted(n for n in todos if n.endswith('.csv.gz'))
    print(f'\n  Archivos totales encontrados: {len(todos)}')
    print(f'  Archivos .csv.gz (datos SEPA): {len(csvgz)}')
    if csvgz:
        for n in csvgz[:20]:
            print(f'    - {n}')
    elif todos:
        print('  ⚠ Ningún .csv.gz encontrado. Primeros archivos hallados:')
        for n in list(todos.keys())[:20]:
            print(f'    - {n}')
    else:
        print('  ⚠ Las carpetas parecen vacías o sin acceso con esta cuenta de Google.')

    meses_completos = sorted(m for m, v in meses_drive.items() if 'parte1' in v and 'parte2' in v)
    if not meses_completos:
        raise RuntimeError(
            f'No se encontraron archivos SEPA en las carpetas de Drive.\n'
            f'  Archivos totales: {len(todos)} | .csv.gz: {len(csvgz)}\n'
            f'  Patrón esperado: MMAAAA_pais_parteNCOMPLETO.csv.gz\n'
            f'  Verificá que:\n'
            f'    1. Iniciaste sesión en Colab con la cuenta que tiene acceso a esas carpetas\n'
            f'    2. Los archivos están en las carpetas correctas (o en subcarpetas)\n'
            f'    3. Los nombres de archivo siguen el patrón esperado (ver arriba)'
        )

    MES_ANALIZAR = MES if (MES and MES in meses_drive) else meses_completos[-1]
    mm   = MES_ANALIZAR[5:]
    aaaa = MES_ANALIZAR[:4]
    MES_MMAAAA = f'{mm}{aaaa}'

    print(f'\n  Meses con datos completos: {meses_completos}')
    print(f'  Mes seleccionado: {MES_ANALIZAR}')

    def descargar_drive(file_id, output_path, label):
        p = Path(output_path)
        if p.exists() and p.stat().st_size > 100_000:
            print(f'  ↩ {label} ya descargado')
            return
        print(f'  ↓ {label}...')
        req = drive.files().get_media(fileId=file_id)
        with open(output_path, 'wb') as f:
            dl = MediaIoBaseDownload(f, req, chunksize=50*1024*1024)
            done = False
            while not done:
                status, done = dl.next_chunk()
                if status:
                    print(f'    {int(status.progress()*100)}%', end='\r')
        print(f'  ✓ {label}: {Path(output_path).stat().st_size/1e6:.0f} MB')

    descargar_drive(meses_drive[MES_ANALIZAR]['parte1']['id'],
                    f'{DIR_DATOS}/{MES_MMAAAA}_pais_parte1COMPLETO.csv.gz', 'SEPA parte 1')
    descargar_drive(meses_drive[MES_ANALIZAR]['parte2']['id'],
                    f'{DIR_DATOS}/{MES_MMAAAA}_pais_parte2COMPLETO.csv.gz', 'SEPA parte 2')

    for nombre in (NOMBRE_MAESTRO_PROD, NOMBRE_MAESTRO_SUC):
        if nombre in todos:
            descargar_drive(todos[nombre]['id'], f'{DIR_DATOS}/{nombre}', nombre)
        else:
            print(f'  ⚠ No se encontró "{nombre}" en Drive')

    # Mostrar IDs descubiertos para que se puedan agregar al manifest
    print('\n' + '='*60)
    print('IDs descubiertos — pegá esto en data/drive_manifest.json:')
    print('='*60)
    ids_desc = {
        'meses': {MES_ANALIZAR: {
            'parte1_id': meses_drive[MES_ANALIZAR]['parte1']['id'],
            'parte2_id': meses_drive[MES_ANALIZAR]['parte2']['id'],
        }},
        'maestros': {
            'productos_id':  todos.get(NOMBRE_MAESTRO_PROD, {}).get('id', 'no encontrado'),
            'sucursales_id': todos.get(NOMBRE_MAESTRO_SUC,  {}).get('id', 'no encontrado'),
        }
    }
    print(json.dumps(ids_desc, indent=2))
    print('='*60)

print(f'\n✓ Archivos listos en {DIR_DATOS}/')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# PASO 4 — Cargar y limpiar datos SEPA
# ══════════════════════════════════════════════════════════════════
import gc
import pandas as pd
from pathlib import Path

from sepa.pipeline.loader import iter_semester_csvgz, load_master_branches
from sepa.pipeline.cleaner import clean_prices
from sepa.pipeline.enricher import enrich_with_branches, filter_excluded_chains
from sepa.config.canasta import CANASTA_EANS

print(f'Cargando precios de {MES_ANALIZAR}...')
frames = []
for chunk in iter_semester_csvgz(Path(DIR_DATOS), ean_filter=CANASTA_EANS):
    chunk = filter_excluded_chains(chunk)
    chunk = clean_prices(chunk, auto_scale=True)
    frames.append(chunk)
    gc.collect()

if not frames:
    raise RuntimeError('Sin datos. Verificá que los CSV.GZ se descargaron correctamente.')

df_long = pd.concat(frames, ignore_index=True)
del frames; gc.collect()

print(f'✓ {len(df_long):,} registros | {df_long["ean_norm"].nunique()} EANs de canasta')
print(f'  Período: {df_long["fecha"].min().date()} → {df_long["fecha"].max().date()}')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# PASO 5 — Enriquecer con maestros
# ══════════════════════════════════════════════════════════════════
mb = load_master_branches(f'{DIR_DATOS}/{NOMBRE_MAESTRO_SUC}')
df_enriched = enrich_with_branches(df_long, mb)
del df_long; gc.collect()

branch_cols = [c for c in ['id_comercio','id_bandera','id_sucursal'] if c in df_enriched.columns]
print(f'✓ {len(df_enriched[branch_cols].drop_duplicates()):,} sucursales | '
      f'{df_enriched.get("provincia", pd.Series(dtype=str)).nunique()} provincias | '
      f'{df_enriched.get("cadena", pd.Series(dtype=str)).nunique()} cadenas')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# PASO 6 — Canasta mensual por sucursal
# ══════════════════════════════════════════════════════════════════
from sepa.pipeline.aggregator import compute_monthly_avg, build_branch_basket

df_monthly = compute_monthly_avg(df_enriched)
df_branch  = build_branch_basket(df_monthly)

df_mes = df_branch[df_branch['mes'] == MES_ANALIZAR].copy()
if df_mes.empty:
    raise RuntimeError(f'Sin datos para {MES_ANALIZAR}. Meses en los datos: {sorted(df_branch["mes"].unique())}')

avg = df_mes['canasta_total'].mean()
cv  = df_mes['canasta_total'].std() / avg * 100
print(f'══ ICM-UADE — {MES_ANALIZAR} ══')
print(f'Sucursales:       {len(df_mes):>8,}')
print(f'Canasta promedio: ${avg:>12,.0f}'.replace(',','.'))
print(f'Canasta mediana:  ${df_mes["canasta_total"].median():>12,.0f}'.replace(',','.'))
print(f'Dispersión (CV):  {cv:>11.1f}%')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# PASO 7 — Mapa interactivo
# ══════════════════════════════════════════════════════════════════
from sepa.viz.maps import make_branch_map
from IPython.display import IFrame

suc_cols = branch_cols + [c for c in ['lat','lng','cadena','provincia','region',
                                       'sucursales_nombre','localidad_nombre','barrio_caba']
                           if c in df_enriched.columns]
df_suc = df_enriched[suc_cols].drop_duplicates(subset=branch_cols)

mapa_path = f'/content/mapa_canasta_pais_{MES_MMAAAA}_filtros.html'
make_branch_map(df_mes, df_suc, mapa_path, mes=MES_ANALIZAR)
print(f'✓ Mapa generado ({Path(mapa_path).stat().st_size/1e6:.0f} MB)')
IFrame(mapa_path, width='100%', height=600)

In [ ]:
# ══════════════════════════════════════════════════════════════════
# PASO 8 — Rankings de cadenas
# ══════════════════════════════════════════════════════════════════
from sepa.analysis.chains import national_ranking, amba_ranking
from sepa.viz.charts import plot_chain_ranking
from IPython.display import Image, display as ipy_display

rank_nac  = national_ranking(df_mes, df_suc, mes=MES_ANALIZAR)
rank_amba = amba_ranking(df_mes, df_suc, mes=MES_ANALIZAR)

print('── Ranking Nacional ──')
display(rank_nac[['ranking','cadena','canasta_promedio','n_sucursales','vs_promedio_pct']]
        .style.format({'canasta_promedio':'${:,.0f}','vs_promedio_pct':'{:+.1f}%'}).hide(axis='index'))

png_nac = f'/content/ranking_cadenas_nacional_{MES_MMAAAA}.png'
plot_chain_ranking(rank_nac, png_nac, title=f'Ranking Nacional — {MES_ANALIZAR}')
ipy_display(Image(png_nac))

if not rank_amba.empty:
    print('\n── Ranking AMBA ──')
    display(rank_amba[['ranking','cadena','canasta_promedio','n_sucursales','vs_promedio_pct']]
            .style.format({'canasta_promedio':'${:,.0f}','vs_promedio_pct':'{:+.1f}%'}).hide(axis='index'))
    png_amba = f'/content/ranking_cadenas_amba_{MES_MMAAAA}.png'
    plot_chain_ranking(rank_amba, png_amba, title=f'Ranking AMBA — {MES_ANALIZAR}')
    ipy_display(Image(png_amba))

In [ ]:
# ══════════════════════════════════════════════════════════════════
# PASO 9 — Ranking provincial
# ══════════════════════════════════════════════════════════════════
from sepa.pipeline.aggregator import aggregate_by_province
from sepa.analysis.basket import basket_by_province
from sepa.viz.charts import plot_province_ranking

if 'provincia' in df_suc.columns:
    df_prov = aggregate_by_province(df_mes, df_suc)
    df_rank_prov = basket_by_province(df_prov, MES_ANALIZAR)
    print('── Ranking Provincial ──')
    display(df_rank_prov[['ranking','provincia','canasta_provincia','vs_promedio_pct']]
            .style.format({'canasta_provincia':'${:,.0f}','vs_promedio_pct':'{:+.1f}%'}).hide(axis='index'))
    png_prov = f'/content/ranking_provincias_{MES_MMAAAA}.png'
    plot_province_ranking(df_prov, png_prov, mes=MES_ANALIZAR)
    ipy_display(Image(png_prov))

In [ ]:
# ══════════════════════════════════════════════════════════════════
# PASO 10 — Ranking barrios CABA
# ══════════════════════════════════════════════════════════════════
from sepa.analysis.basket import barrio_ranking_caba

df_barrios = barrio_ranking_caba(df_mes, df_suc, mes=MES_ANALIZAR)
if not df_barrios.empty:
    print(f'── Ranking Barrios CABA ({len(df_barrios)} barrios) ──')
    display(df_barrios[['ranking','barrio_caba','canasta_barrio','n_sucursales','vs_promedio_caba_pct']]
            .rename(columns={'barrio_caba':'barrio','canasta_barrio':'canasta','vs_promedio_caba_pct':'vs CABA %'})
            .style.format({'canasta':'${:,.0f}','vs CABA %':'{:+.1f}%'}).hide(axis='index'))

In [ ]:
# ══════════════════════════════════════════════════════════════════
# PASO 11 — Descargar outputs a tu computadora
# ══════════════════════════════════════════════════════════════════
from google.colab import files

outputs = [
    (f'/content/mapa_canasta_pais_{MES_MMAAAA}_filtros.html', 'mapa HTML interactivo'),
    (f'/content/ranking_cadenas_nacional_{MES_MMAAAA}.png',   'ranking nacional PNG'),
    (f'/content/ranking_cadenas_amba_{MES_MMAAAA}.png',       'ranking AMBA PNG'),
    (f'/content/ranking_provincias_{MES_MMAAAA}.png',          'ranking provincial PNG'),
]

print(f'Descargando outputs de {MES_ANALIZAR} a tu computadora...')
for ruta, label in outputs:
    if Path(ruta).exists():
        files.download(ruta)
        print(f'  ✓ {label}')
    else:
        print(f'  ⚠ No generado: {label}')

print(f'\n✓ Análisis de {MES_ANALIZAR} completado.')